<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>LLM Engineering Masterclass</h1>
<h1>5. Tool-Using Agents with Guardrails</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import os
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import watermark
from pydantic import BaseModel, ValidationError

# Data downloads/cache and figure output locations
_ROOT = Path.cwd()
_DATA = _ROOT / "data"
_OUT = _ROOT / "output"
_DATA.mkdir(parents=True, exist_ok=True)
_OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(_DATA / "huggingface"))
os.environ.setdefault("HF_DATASETS_CACHE", str(_DATA / "huggingface" / "datasets"))
os.environ.setdefault("KAGGLEHUB_CACHE", str(_DATA / "kagglehub"))

from collections import Counter
from pprint import pprint

%load_ext watermark
%matplotlib inline

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.14.5
IPython version      : 9.17.1

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: a1c986090a5c99e2014c7651bb71d8130e7ae0fc

json         : 2.0.9
matplotlib   : 3.11.1
numpy        : 2.5.3
pandas       : 3.0.5
pydantic     : 2.13.5
pydantic_core: 2.46.5
watermark    : 2.6.0



In [3]:
plt.style.use("d4sci.mplstyle")
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# A loop with power tools

An agent is a loop. The model picks a tool, the harness runs it, the result feeds back in, repeat until the model answers or a budget runs out.

The loop is also the danger. One wrong step feeds the next, so errors compound. Everything in this notebook exists to break that compounding:

1. Validated arguments
2. Errors returned as data, never as crashes
3. Policy gates on risky actions
4. Step budgets, retries and fallbacks

The customer messages are real Banking77 texts. The bank behind the tools is a small mock.

# Real messages from the test split

In [4]:
test = pd.read_csv(_DATA / "banking77_test.csv")

def real_message(intent, n=0):
    """A real customer message from the test split for a given intent."""
    return test.loc[test["intent"] == intent, "text"].iloc[n]

print(real_message("transaction_charged_twice"))
print(real_message("lost_or_stolen_card"))

I got double charged for a payment so how do I fix that?
My entire gym bag, including my wallet, was stolen out of my locker today. Everything in my wallet is gone - how do I block the card to make it can't be used?


# The world the agent acts on

In [5]:
# A tiny in-memory core banking system. In production these are real service calls.
CUSTOMERS = {
    "CUST-001": {"name": "Jane Rivera", "email": "jane@example.com", "card_id": "CARD-9001"},
    "CUST-002": {"name": "Sam Ortiz", "email": "sam@example.com", "card_id": "CARD-9002"},
}

TRANSACTIONS = {
    "TX-1001": {"customer_id": "CUST-001", "amount": 42.50, "merchant": "Coffee & Co", "date": "2026-09-10", "status": "settled"},
    "TX-1002": {"customer_id": "CUST-001", "amount": 42.50, "merchant": "Coffee & Co", "date": "2026-09-10", "status": "settled"},
    "TX-1003": {"customer_id": "CUST-001", "amount": 310.00, "merchant": "Rail Tickets", "date": "2026-09-08", "status": "settled"},
    "TX-1004": {"customer_id": "CUST-002", "amount": 9.99, "merchant": "Music Stream", "date": "2026-09-09", "status": "pending"},
}

CARDS = {
    "CARD-9001": {"customer_id": "CUST-001", "status": "active"},
    "CARD-9002": {"customer_id": "CUST-002", "status": "active"},
}

REFUND_LOG = []

def find_customer(email):
    for customer_id, data in CUSTOMERS.items():
        if data["email"] == email:
            return {"customer_id": customer_id, **data}
            
    raise LookupError(f"No customer with email {email}")

def list_transactions(customer_id):
    return [{"transaction_id": key, **value}
            for key, value in TRANSACTIONS.items()
            if value["customer_id"] == customer_id]

def refund_transaction(transaction_id, reason):
    transaction = TRANSACTIONS[transaction_id]
    
    if transaction["amount"] > 100:
        return {"status": "requires_human_approval",
                "detail": "Refunds above 100 need a human sign off."}
    
    REFUND_LOG.append({"transaction_id": transaction_id, "amount": transaction["amount"], "reason": reason})
    
    return {"status": "refunded", "amount": transaction["amount"]}

# Tools with contracts

Each tool gets a Pydantic argument model. One definition gives us runtime validation and the JSON schema the API needs. Write it once.

In [6]:
class FindCustomerArgs(BaseModel):
    email: str

class ListTransactionsArgs(BaseModel):
    customer_id: str

class RefundTransactionArgs(BaseModel):
    transaction_id: str
    reason: str

TOOLS = {
    "find_customer": (find_customer, FindCustomerArgs, "Look up a customer by email."),
    "list_transactions": (list_transactions, ListTransactionsArgs, "List recent card transactions for a customer id."),
    "refund_transaction": (refund_transaction, RefundTransactionArgs, "Refund one transaction by id."),
}

def tool_specs():
    specs = []
    for name, (fn, args_model, description) in TOOLS.items():
        specs.append({
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        })
        
    return specs

tool_specs()[0]

{'type': 'function',
 'function': {'name': 'find_customer',
  'description': 'Look up a customer by email.',
  'parameters': {'properties': {'email': {'title': 'Email', 'type': 'string'}},
   'required': ['email'],
   'title': 'FindCustomerArgs',
   'type': 'object'}}}

# The dispatcher: where crashes go to die

The model will pass bad arguments. Services will time out. Neither may kill the loop. The dispatcher turns every failure into a message the model can read and react to.

In [7]:
def run_tool(name, raw_args):
    """Validate first. Never let a tool exception kill the loop. Errors are data."""
    
    if name not in TOOLS:
        return {"error": f"Unknown tool {name}"}

    fn, args_model, _ = TOOLS[name]

    try:
        args = args_model.model_validate_json(raw_args)
    except ValidationError as error:
        return {"error": f"Invalid arguments: {error}"}

    try:
        return fn(**args.model_dump())
    except Exception as error:
        return {"error": f"{type(error).__name__}: {error}"}

# The agent loop

In [8]:
from openai import OpenAI

openai_client = OpenAI()
DEFAULT_MODEL = "gpt-4.1-mini"

AGENT_SYSTEM = (
    "You are a support agent for a mobile banking app.\n"
    "The customer message arrives with the sender's email. Use the tools to investigate before you act.\n"
    "Refund only clear errors, such as the same merchant charging the same amount twice on one day.\n"
    "If a tool reports requires_human_approval, tell the customer a colleague will follow up.\n"
    "When you are done, reply with a short message for the customer."
)

def run_agent(user_message, model=DEFAULT_MODEL, max_steps=8):
    messages = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": user_message},
    ]
    
    for step in range(max_steps):
        response = openai_client.chat.completions.create(
            model=model, 
            messages=messages, 
            tools=tool_specs(), 
            temperature=0,
        )
        
        message = response.choices[0].message
        
        if not message.tool_calls:
            print(f"\nAgent: {message.content}")
            return message.content
            
        messages.append(message)

        for tool_call in message.tool_calls:
            result = run_tool(tool_call.function.name, tool_call.function.arguments)
            
            print(f"[step {step}] {tool_call.function.name}({tool_call.function.arguments})")
            print(f"         -> {json.dumps(result)[:140]}")
            
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                             "content": json.dumps(result)})

    print("\nAgent: I hit my step budget. Handing this off to a human.")
    
    return None

# Run 1: a real double charge report

Jane's account holds two identical coffee charges on the same day. The refund is small, so the policy allows it. Watch the transcript.

In [9]:
incoming = f"From jane@example.com: {real_message('transaction_charged_twice')}"
print(incoming)

transcript = run_agent(incoming)
print("\nRefund log:", REFUND_LOG)

From jane@example.com: I got double charged for a payment so how do I fix that?
[step 0] find_customer({"email":"jane@example.com"})
         -> {"customer_id": "CUST-001", "name": "Jane Rivera", "email": "jane@example.com", "card_id": "CARD-9001"}
[step 1] list_transactions({"customer_id":"CUST-001"})
         -> [{"transaction_id": "TX-1001", "customer_id": "CUST-001", "amount": 42.5, "merchant": "Coffee & Co", "date": "2026-09-10", "status": "settle
[step 2] refund_transaction({"transaction_id":"TX-1002","reason":"Duplicate charge for the same merchant and amount on the same day"})
         -> {"status": "refunded", "amount": 42.5}

Agent: I have processed a refund for the duplicate charge from Coffee & Co on your account. You should see the refunded amount of $42.50 reflected soon. If you need any more help, feel free to ask!

Refund log: [{'transaction_id': 'TX-1002', 'amount': 42.5, 'reason': 'Duplicate charge for the same merchant and amount on the same day'}]


# Run 2: the policy gate

Same customer, a bigger ask. The rail tickets cost 310, above the refund gate. The model decides what to do. Code decides what it may do.

In [10]:
incoming = "From jane@example.com: I want my money back for the rail tickets, the train never came."
print(incoming)

transcript = run_agent(incoming)

From jane@example.com: I want my money back for the rail tickets, the train never came.
[step 0] find_customer({"email":"jane@example.com"})
         -> {"customer_id": "CUST-001", "name": "Jane Rivera", "email": "jane@example.com", "card_id": "CARD-9001"}
[step 1] list_transactions({"customer_id":"CUST-001"})
         -> [{"transaction_id": "TX-1001", "customer_id": "CUST-001", "amount": 42.5, "merchant": "Coffee & Co", "date": "2026-09-10", "status": "settle

Agent: I see the transaction for the rail tickets on your account. However, since the train never came, I will need to escalate this issue to a colleague for further investigation and follow-up. They will contact you soon to assist with your refund request.


# Retries: for reads, not writes

Transient failures deserve a second try. Business actions do not, at least not blindly. Retry a lookup freely. Retry a refund and you may issue it twice. Writes need idempotency keys or a state check before the retry.

In [11]:
def with_retries(fn, attempts=3, base_delay=1.0):
    """Exponential backoff for transient failures: 429s, 5xx, timeouts."""
    def wrapped(*args, **kwargs):
        for attempt in range(attempts):
            try:
                return fn(*args, **kwargs)
            except Exception as error:
                if attempt == attempts - 1:
                    raise
                delay = base_delay * 2 ** attempt
                print(f"Retry {attempt + 1} after {type(error).__name__}. Sleeping {delay:.0f}s.")
                time.sleep(delay)

    return wrapped

# Failure injection

Do not wait for production to learn how your agent fails. Break a tool on purpose and watch the recovery.

In [12]:
original_find_customer = find_customer

def flaky_find_customer(email):
    """Simulates a core banking outage for one unlucky email."""
    if email.startswith("eve@"):
        raise TimeoutError("Customer lookup timed out")
    return original_find_customer(email)

TOOLS["find_customer"] = (flaky_find_customer, FindCustomerArgs, "Look up a customer by email.")

incoming = f"From eve@example.com: {real_message('transaction_charged_twice', n=1)}"
print(incoming)

transcript = run_agent(incoming)

From eve@example.com: I've just come back from an eating holiday in the USA and Canada and have SO many pending and duplicated transactions on my account.  I think there is something wrong, can you look in to it please?
[step 0] find_customer({"email":"eve@example.com"})
         -> {"error": "TimeoutError: Customer lookup timed out"}

Agent: I am having trouble accessing your account information at the moment. Could you please try again in a little while? If the issue persists, I will escalate it for further assistance.


The timeout came back as data. The agent apologized and asked for another route instead of hallucinating a customer record. Graceful degradation is a design outcome, not luck.

### Exercise: a dangerous tool, done safely

Add `freeze_card(card_id, confirm)` for the `lost_or_stolen_card` messages in the test split. Requirements: `confirm` must be true or the tool refuses, and the system prompt must direct the agent to get explicit customer confirmation first. Test both paths with `real_message('lost_or_stolen_card')`.

In [13]:
# Your code here
# class FreezeCardArgs(BaseModel): ...


# Recap

1. Validate every argument. Pydantic makes the contract executable.
2. Return errors as data. The loop survives and the model adapts.
3. Gate risky actions in code. Approval walls beat clever prompts.
4. Cap steps. Retry reads. Guard writes with idempotency.

The agent works. Next: watching it work in production, where the refunds are real.

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>